#Load Data

In [1]:
chunks = [

    "Employees receive 12 casual leaves annually.",

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week.",

    "Travel expenses are reimbursed within 30 days.",

    "All employees are covered under company medical insurance."

]

print("Total Chunks:", len(chunks))

Total Chunks: 5


#Load Embedding Model

In [3]:
from sentence_transformers import SentenceTransformer

In [4]:
model=SentenceTransformer(
    "all-MiniLM-L6-v2"
)
print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


#Generate Embeddings

In [5]:
embeddings=model.encode(chunks)
print("Embeddings Generated")
print("\nEmbeddings shape:",embeddings.shape)

Embeddings Generated

Embeddings shape: (5, 384)


#Create FAISS Index

In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 25.8 MB/s eta 0:00:00


In [9]:
import faiss

dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
print("FAISS Index Created")

FAISS Index Created


#Add Embeddings to FAISS

In [10]:
import numpy as np
index.add(
    np.array(embeddings,
             dtype=np.float32)
)
print("Vectors Stored:",index.ntotal)

Vectors Stored: 5


#User Query

In [12]:
query="How many casual leaves are allowed?"
print(query)

How many casual leaves are allowed?


#Covert Query into Embeddings

In [13]:
query_embedding=model.encode(
    [query]
)

#Search FAISS

In [15]:
k=3
distances,indices=index.search(
    np.array(
        query_embedding,
        dtype=np.float32
    ),
    k
    )

#Display results

In [16]:
print("Matching Chunks\n")
for rank,idx in enumerate(indices[0]):
  print(f'Rank {rank+1}:')
  print(chunks[idx])
  print("Distance:",distances[0][rank])
  print("-"*50)

Matching Chunks

Rank 1:
Employees receive 12 casual leaves annually.
Distance: 0.68964183
--------------------------------------------------
Rank 2:
Employees receive 15 sick leaves annually.
Distance: 1.1779994
--------------------------------------------------
Rank 3:
Employees may work from home twice per week.
Distance: 1.6474638
--------------------------------------------------
